# Phase 2B — target→S1 multikey blocker

Thin CPU launcher. Attach the raw dataset and Phase-1 normalized output. The launcher prefers read-only `/kaggle/input`; a live-session `/kaggle/working` fallback is provided. **UNVERIFIED runtime estimate:** 20–90 minutes.

In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys
REPO_URL='https://github.com/yugtheguy/NEW_AMAZON.git'
PINNED_COMMIT='ec36fea31732632ba53b6fafd2e60d5ced35600d'
INPUT_ROOT=Path('/kaggle/input'); REPO_DIR=Path('/kaggle/working/NEW_AMAZON'); RESUME_ARTIFACTS_ROOT=None

In [ ]:
expected={'train_source1.tsv','train_source2.tsv','train_source3.tsv','train_ground_truth.tsv','test_source1.tsv','test_source2.tsv','test_source3.tsv'}
found={name:list(INPUT_ROOT.rglob(name)) for name in expected}; problems={name:p for name,p in found.items() if len(p)!=1}
if problems: raise RuntimeError(f'Raw inputs must occur exactly once: {problems}')
manifests=[]
for path in INPUT_ROOT.rglob('part-*.manifest.json'):
    try:
        if json.loads(path.read_text(encoding='utf-8')).get('stage')=='normalization': manifests.append(path)
    except Exception: pass
roots=sorted({p.parents[3] for p in manifests})
working=Path('/kaggle/working/NEW_AMAZON/artifacts/normalized/v1')
if len(roots)==1: NORMALIZED_ROOT=roots[0]
elif not roots and working.is_dir(): NORMALIZED_ROOT=working; print('WARNING: using live-session writable normalization artifacts')
else: raise RuntimeError(f'Expected one normalization_v1 root: {roots}')
assert len(list(NORMALIZED_ROOT.rglob('part-*.parquet')))==89
assert len(list(NORMALIZED_ROOT.rglob('part-*.manifest.json')))==89
print('Normalized root:',NORMALIZED_ROOT)

In [ ]:
if not REPO_DIR.exists(): subprocess.run(['git','clone','--filter=blob:none',REPO_URL,str(REPO_DIR)],check=True)
else: subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin'],check=True)
subprocess.run(['git','-C',str(REPO_DIR),'checkout','--detach',PINNED_COMMIT],check=True)
actual=subprocess.check_output(['git','-C',str(REPO_DIR),'rev-parse','HEAD'],text=True).strip(); assert actual==PINNED_COMMIT
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{REPO_DIR}[data]','duckdb==1.5.5'],check=True)
help_text=subprocess.check_output([sys.executable,'-m','amazon_er.cli','run','--help'],text=True); assert '--normalized-root' in help_text
print('Active commit:',actual)

In [ ]:
if RESUME_ARTIFACTS_ROOT is not None:
    source=Path(RESUME_ARTIFACTS_ROOT)/'retrieval/v1/multikey_blocker'; destination=REPO_DIR/'artifacts/retrieval/v1/multikey_blocker'
    if source.exists(): shutil.copytree(source,destination,dirs_exist_ok=True)
subprocess.run(['bash','-lc','free -h && df -h /kaggle/working'],check=True)
log_path=Path('/kaggle/working/phase2b_multikey.log')
command=[sys.executable,'-m','amazon_er.cli','run','--stage','retrieve-multikey','--config',str(REPO_DIR/'configs/prod.yaml'),'--normalized-root',str(NORMALIZED_ROOT),'--data-root',str(INPUT_ROOT)]
with log_path.open('a',encoding='utf-8') as log:
    process=subprocess.Popen(command,cwd=REPO_DIR,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in process.stdout: print(line,end=''); log.write(line); log.flush()
    code=process.wait()
if code: raise RuntimeError(f'Phase 2B failed with exit code {code}; inspect {log_path}')

In [ ]:
root=REPO_DIR/'artifacts/retrieval/v1/multikey_blocker'; report=REPO_DIR/'artifacts/reports/retrieval/v1/multikey_blocker/summary.json'
parts=list(root.rglob('part-*.parquet')); manifests=list(root.rglob('part-*.manifest.json')); assert parts and len(parts)==len(manifests)
summary=json.loads(report.read_text(encoding='utf-8')); assert summary['status']=='complete'; assert summary['raw_key_coverage'] is not None
print({'parquets':len(parts),'manifests':len(manifests)}); print(json.dumps(summary,indent=2,sort_keys=True)); print('Use Save Version now.')